In [5]:
# !pip install roboflow ultralytics

from roboflow import Roboflow
rf = Roboflow(api_key="R3fB4ZYSkiyvcHZgBYN5")
project = rf.workspace("test-3j2z9").project("test-ncscd")
version = project.version(2)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


In [1]:
import os

PATH = './test-2/train/images/'

files = os.listdir(PATH)

len(files)

1037

In [6]:
dataset.location

import yaml

# Path to your data.yaml
yaml_path = f"{dataset.location}/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Pulling the class names
classes = data.get('names')

print(f"Total classes: {len(classes)}")
print("Labels:", classes)

Total classes: 2
Labels: ['2door', 'door']


In [7]:
from pathlib import Path

image_dir = Path(f"{dataset.location}/train/images")
label_dir = Path(f"{dataset.location}/train/labels")

backgrounds = []

for img in image_dir.glob("*"):
    label = label_dir / f"{img.stem}.txt"

    if not label.exists() or label.stat().st_size == 0:
        backgrounds.append(img)

print(f"Background images: {len(backgrounds)}")
print(backgrounds[:10])

Background images: 0
[]


In [8]:
from pathlib import Path
import os

image_dir = Path(f"{dataset.location}/train/images")
label_dir = Path(f"{dataset.location}/train/labels")

removed = 0

for img in image_dir.glob("*"):
    label = label_dir / f"{img.stem}.txt"

    if not label.exists() or label.stat().st_size == 0:
        os.remove(img)

        if label.exists():
            os.remove(label)

        removed += 1

print(f"Removed {removed} background images")

Removed 0 background images


In [9]:
from ultralytics import YOLO
import os
project=os.path.join(os.getcwd(), "../../runs/detect")  # adjust depth to repo root

# 1. Load the YOLO26 Nano model (pretrained on COCO)
# Using the .pt file ensures we are fine-tuning, not training from scratch
model = YOLO('yolo26n.pt')

# 2. Fine-tune the model
# dataset.location was defined when you ran version.download("yolo26")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    project = project,
    epochs=10,
    imgsz=1024,
    batch=4,          # high imgsz eats memory fast; 4-8 is realistic at 1024
    workers=0,
    cache="ram",
    device="mps",
    amp=False,
    rect=True,        # batches images of similar aspect ratio → less padding waste
    plots=True,
)

Ultralytics 8.4.47 🚀 Python-3.12.0 torch-2.11.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/notebooks/model/test-2/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimiz

In [10]:
# Validate the model's performance on the validation set
metrics = model.val()

# Export for deployment (e.g., to ONNX for web/mobile use)
model.export(format='onnx')

Ultralytics 8.4.47 🚀 Python-3.12.0 torch-2.11.0 CPU (Apple M5)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 129.9±31.7 MB/s, size: 32.1 KB)
val: Scanning /Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/notebooks/model/test-2/valid/labels.cache... 133 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 133/133 62.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.2s/it 37.4s4.9ss
                   all        133        863      0.725       0.76      0.755       0.46
                 2door         13         33      0.603      0.606      0.584      0.279
                  door        128        830      0.847      0.914      0.927      0.642
Speed: 2.0ms preprocess, 275.0ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /opt/homebrew/runs/detect/val
Ultralytics 8.4.47 🚀 Python-3.12.0 torch-2.11

Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.


ONNX: export success ✅ 0.7s, saved as '/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/runs/detect/train/weights/best.onnx' (9.6 MB)

Export complete (0.9s)
Results saved to /Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/runs/detect/train/weights/best.onnx
Predict:         yolo predict task=detect model=/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/runs/detect/train/weights/best.onnx imgsz=1024 
Validate:        yolo val task=detect model=/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/runs/detect/train/weights/best.onnx imgsz=1024 data=/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/notebooks/model/test-2/data.yaml  
Visualize:       https://netron.app


'/Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/runs/detect/train/weights/best.onnx'

In [12]:
model = YOLO("../../runs/detect/train/weights/best.pt")

# Run inference
results = model("../../data/PNG/no_text/IIa_A01001.png")

# Show results
results[0].show()

# Save results
results[0].save(filename="result.jpg")


image 1/1 /Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/notebooks/model/../../data/PNG/no_text/IIa_A01001.png: 736x1024 7 doors, 56.2ms
Speed: 3.9ms preprocess, 56.2ms inference, 0.1ms postprocess per image at shape (1, 3, 736, 1024)


'result.jpg'

In [15]:
import cv2
# Load and resize image
image = cv2.imread("../../data/PNG/no_text/IIa_A01001.png")
image_resized = cv2.resize(image, (640, 640))

# Run inference
results = model(image_resized)

# Show results
results[0].show()

# Save annotated result
results[0].save(filename="result.jpg")


0: 1024x1024 9 doors, 94.5ms
Speed: 3.7ms preprocess, 94.5ms inference, 0.1ms postprocess per image at shape (1, 3, 1024, 1024)


'result.jpg'